In [1]:
import pandas as pd
import plotly.express as px


In [2]:
df = pd.read_csv('data/covid_deaths_by_state_cumulative_snapshot.csv')
df.head()

,state,state_abbreviation,census_region,covid_19_deaths,data_as_of,period_start,period_end
0,Alabama,AL,South,23035,08/13/2026,01/01/2020,08/08/2026
1,Alaska,AK,West,1638,08/13/2026,01/01/2020,08/08/2026
2,Arizona,AZ,West,32243,08/13/2026,01/01/2020,08/08/2026
3,Arkansas,AR,South,13662,08/13/2026,01/01/2020,08/08/2026
4,California,CA,West,117655,08/13/2026,01/01/2020,08/08/2026


In [3]:
px.bar(df, x='state', y='covid_19_deaths', title='Cumulative COVID-19 Deaths by State')

In [15]:
px.bar(df, 
       y='state', 
       x='covid_19_deaths', 
       orientation='h',
       title='COVID-19 Deaths by State',
       labels={'Total Deaths': 'Total Deaths', 'State': ''},
       width=500, height=800)

In [14]:
df = df.sort_values(by="covid_19_deaths", ascending=True)

# Same region colors as the chart above: Northeast, Midwest (middle America),
# South (including the Southeast), and West (including the Southwest).
#
# Adding color splits the bars into one series per region. Plotly would then
# bunch the states by color and undo the sort. categoryorder "total ascending"
# puts the ranking back: shortest bar at the bottom, longest at the top.
# dtick=1 keeps every state name on the axis instead of skipping some.
fig = px.bar(
    df,
    orientation="h",
    y="state",
    x="covid_19_deaths",
    color="census_region",
    labels={"census_region": "Region"},
    width=500,
    height=800,
)
fig.update_yaxes(dtick=1, categoryorder="total ascending")
# The legend sits above the bars so it does not squeeze the state names.
fig.update_layout(legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0})
fig


In [6]:
# Resident population for the 50 states and the District of Columbia.
#
# Source: U.S. Census Bureau, Population Division, Vintage 2025
# (NST-EST2025-ALLDATA, column POPESTIMATE2025).
# These are the official estimates for July 1, 2025, released January 2026.
# As of October 2026, the Census Bureau has not published a July 1, 2026
# estimate yet. That next vintage usually comes out between December and
# the following January, so July 1, 2025 is the latest official number
# available during 2026.
#
# Puerto Rico is in the same Census file, but it is a territory, not a
# state, and it is not in our state-level COVID snapshot, so it is left out.
# The District of Columbia is included because our death counts include it.
#
# Keys are full state names so they match the `state` column in the COVID data.
# Values are people, not thousands or millions.
state_population_2025 = {
    "Alabama": 5193088,
    "Alaska": 737270,
    "Arizona": 7623818,
    "Arkansas": 3114791,
    "California": 39355309,
    "Colorado": 6012561,
    "Connecticut": 3688496,
    "Delaware": 1059952,
    "District of Columbia": 693645,
    "Florida": 23462518,
    "Georgia": 11302748,
    "Hawaii": 1432820,
    "Idaho": 2029733,
    "Illinois": 12719141,
    "Indiana": 6973333,
    "Iowa": 3238387,
    "Kansas": 2977220,
    "Kentucky": 4606864,
    "Louisiana": 4618189,
    "Maine": 1414874,
    "Maryland": 6265347,
    "Massachusetts": 7154084,
    "Michigan": 10127884,
    "Minnesota": 5830405,
    "Mississippi": 2954160,
    "Missouri": 6270541,
    "Montana": 1144694,
    "Nebraska": 2018006,
    "Nevada": 3282188,
    "New Hampshire": 1415342,
    "New Jersey": 9548215,
    "New Mexico": 2125498,
    "New York": 20002427,
    "North Carolina": 11197968,
    "North Dakota": 799358,
    "Ohio": 11900510,
    "Oklahoma": 4123288,
    "Oregon": 4273586,
    "Pennsylvania": 13059432,
    "Rhode Island": 1114521,
    "South Carolina": 5570274,
    "South Dakota": 935094,
    "Tennessee": 7315076,
    "Texas": 31709821,
    "Utah": 3538904,
    "Vermont": 644663,
    "Virginia": 8880107,
    "Washington": 8001020,
    "West Virginia": 1766147,
    "Wisconsin": 5972787,
    "Wyoming": 588753,
}

# 50 states + D.C. should be 51. This check fails loudly if a row is missing.
assert len(state_population_2025) == 51


In [9]:
# Look up each row's state name in the population dictionary and
# store the matching number in a new column.
# .map() walks down df["state"] one value at a time. For "Texas" it
# returns 31709821; for "Wyoming" it returns 588753. The result is a
# column lined up with the original rows, so California's population
# stays on California's row.
# A name that is not in the dictionary would become NaN. Every state
# in this snapshot is in the dictionary, so that should not happen.
df["population"] = df["state"].map(state_population_2025)

# Show the new column next to the state name so we can see the join worked.
df.head()


,state,state_abbreviation,census_region,covid_19_deaths,data_as_of,period_start,period_end,population
45,Vermont,VT,Northeast,1271,08/13/2026,01/01/2020,08/08/2026,644663
1,Alaska,AK,West,1638,08/13/2026,01/01/2020,08/08/2026,737270
50,Wyoming,WY,West,2044,08/13/2026,01/01/2020,08/08/2026,588753
11,Hawaii,HI,West,2332,08/13/2026,01/01/2020,08/08/2026,1432820
8,District of Columbia,DC,South,2373,08/13/2026,01/01/2020,08/08/2026,693645


In [10]:
df['death_rate_per_100k'] = df['covid_19_deaths'] / df['population'] * 100_000
df.head()

,state,state_abbreviation,census_region,covid_19_deaths,data_as_of,period_start,period_end,population,death_rate_per_100k
45,Vermont,VT,Northeast,1271,08/13/2026,01/01/2020,08/08/2026,644663,197.157274
1,Alaska,AK,West,1638,08/13/2026,01/01/2020,08/08/2026,737270,222.170982
50,Wyoming,WY,West,2044,08/13/2026,01/01/2020,08/08/2026,588753,347.174452
11,Hawaii,HI,West,2332,08/13/2026,01/01/2020,08/08/2026,1432820,162.755964
8,District of Columbia,DC,South,2373,08/13/2026,01/01/2020,08/08/2026,693645,342.105832


In [12]:
df = df.sort_values(by="death_rate_per_100k", ascending=True)

# One bar per state, colored by the same four Census regions:
# Northeast, Midwest (middle America), South (including the Southeast),
# and West (including the Southwest).
#
# dtick=1 forces every state name onto the y-axis. Without it, Plotly drops
# labels when 51 names do not fit. categoryorder "total ascending" keeps the
# sort after coloring, because color would otherwise group states by region.
fig = px.histogram(
    df,
    x="death_rate_per_100k",
    y="state",
    color="census_region",
    labels={"census_region": "Region"},
    width=500,
    height=800,
)
fig.update_yaxes(dtick=1, categoryorder="total ascending")
fig.update_layout(legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0})
fig


In [13]:
# Choropleth = a map where each state is filled with a color.
# We color by death_rate_per_100k so big states and small states compare fairly.
#
# locations="state_abbreviation"  -> Plotly needs two-letter codes (CA, TX), not full names.
# locationmode="USA-states"       -> those codes are U.S. states + D.C., not country codes.
# scope="usa"                     -> zoom to the United States instead of the whole world.
# color="death_rate_per_100k"     -> the number that drives the fill color on each state.
# color_continuous_scale="Reds"   -> light pink = fewest deaths per 100k, dark red = most.
#                                    Plotly adds a color bar on the side so you can read the scale.
# hover_name / hover_data           -> what shows when you point at a state.
fig = px.choropleth(
    df,
    locations="state_abbreviation",
    locationmode="USA-states",
    scope="usa",
    color="death_rate_per_100k",
    color_continuous_scale="Reds",
    hover_name="state",
    hover_data={
        "covid_19_deaths": ":,",
        "population": ":,",
        "census_region": True,
        "state_abbreviation": False,
        "death_rate_per_100k": ":.1f",
    },
    title="COVID-19 Deaths per 100,000 People by State",
    labels={
        "death_rate_per_100k": "Deaths per 100,000",
        "covid_19_deaths": "Total deaths",
        "population": "Population",
        "census_region": "Region",
    },
)

# Make the color bar a bit easier to read in class.
fig.update_layout(
    coloraxis_colorbar=dict(
        title="Deaths per 100,000",
        ticksuffix="",
    ),
    margin=dict(l=0, r=0, t=50, b=0),
)
fig
